# 02 — Silver Transformation: Connected Vehicle Telemetry

This notebook reads raw data from the **Bronze** lakehouse, applies cleaning,
deduplication, and data quality rules, then writes curated Delta tables to **Silver**.

**Transformations:**
- Remove duplicates by natural key
- Null handling and range validation
- Schema conformance and type casting
- Data quality flags

**Medallion Layer:** Silver (cleaned, conformed)

In [ ]:
# Silver layer Spark config: balanced read/write with V-Order
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, row_number, avg, count,
    datediff, to_date, expr, trim, upper, coalesce
)
from pyspark.sql.window import Window

In [ ]:
# Read Bronze tables from the Bronze lakehouse using absolute path
# NOTE: These IDs are replaced at deploy time by scripts/deploy.ps1
BRONZE_BASE = "abfss://{{WORKSPACE_ID}}@onelake.dfs.fabric.microsoft.com/{{BRONZE_LAKEHOUSE_ID}}"

df_raw_vehicles = spark.read.format("delta").load(f"{BRONZE_BASE}/Tables/raw_vehicles")
df_raw_telemetry = spark.read.format("delta").load(f"{BRONZE_BASE}/Tables/raw_telemetry")
df_raw_dtcs = spark.read.format("delta").load(f"{BRONZE_BASE}/Tables/raw_dtc_events")
df_raw_maintenance = spark.read.format("delta").load(f"{BRONZE_BASE}/Tables/raw_maintenance")

print(f"Bronze raw_vehicles:    {df_raw_vehicles.count()}")
print(f"Bronze raw_telemetry:   {df_raw_telemetry.count()}")
print(f"Bronze raw_dtc_events:  {df_raw_dtcs.count()}")
print(f"Bronze raw_maintenance: {df_raw_maintenance.count()}")

## Clean & Deduplicate Vehicles

In [ ]:
# Deduplicate by VIN, keep latest by mileage
w_vehicle = Window.partitionBy("vin").orderBy(col("mileage").desc())

df_vehicles_clean = (
    df_raw_vehicles
    .withColumn("_rn", row_number().over(w_vehicle))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .withColumn("vin", upper(trim(col("vin"))))
    .filter(col("vin").isNotNull() & (col("vin") != ""))
    .withColumn("mileage", when(col("mileage") < 0, lit(0)).otherwise(col("mileage")))
    .withColumn("_cleaned_at", current_timestamp())
)

df_vehicles_clean.write.format("delta").mode("overwrite").save("Tables/dim_vehicles")
print(f"Silver dim_vehicles: {df_vehicles_clean.count()} rows")

## Clean & Validate Telemetry Readings

In [ ]:
# Deduplicate by reading_id
w_telemetry = Window.partitionBy("reading_id").orderBy(col("timestamp").desc())

df_telemetry_clean = (
    df_raw_telemetry
    .withColumn("_rn", row_number().over(w_telemetry))
    .filter(col("_rn") == 1)
    .drop("_rn")
    # Range validation — flag out-of-range values
    .withColumn("engine_temp_valid",
        when((col("engine_temp_f") >= 100) & (col("engine_temp_f") <= 300), lit(True))
        .otherwise(lit(False)))
    .withColumn("oil_pressure_valid",
        when((col("oil_pressure_psi") >= 0) & (col("oil_pressure_psi") <= 80), lit(True))
        .otherwise(lit(False)))
    .withColumn("battery_voltage_valid",
        when((col("battery_voltage_v") >= 9.0) & (col("battery_voltage_v") <= 16.0), lit(True))
        .otherwise(lit(False)))
    .withColumn("rpm_valid",
        when((col("rpm") >= 0) & (col("rpm") <= 8000), lit(True))
        .otherwise(lit(False)))
    # Anomaly detection flags
    .withColumn("engine_temp_anomaly",
        when(col("engine_temp_f") > 220, lit(True)).otherwise(lit(False)))
    .withColumn("oil_pressure_anomaly",
        when(col("oil_pressure_psi") < 20, lit(True)).otherwise(lit(False)))
    .withColumn("battery_anomaly",
        when(col("battery_voltage_v") < 11.0, lit(True)).otherwise(lit(False)))
    # Overall quality flag
    .withColumn("_data_quality",
        when(
            col("engine_temp_valid") & col("oil_pressure_valid") &
            col("battery_voltage_valid") & col("rpm_valid"),
            lit("PASS")
        ).otherwise(lit("FAIL")))
    .withColumn("_cleaned_at", current_timestamp())
)

df_telemetry_clean.write.format("delta").mode("overwrite").save("Tables/fact_telemetry")

total = df_telemetry_clean.count()
passed = df_telemetry_clean.filter(col("_data_quality") == "PASS").count()
print(f"Silver fact_telemetry: {total} rows ({passed} passed quality, {total - passed} flagged)")

## Clean DTC Events

In [ ]:
# Deduplicate by dtc_event_id
w_dtc = Window.partitionBy("dtc_event_id").orderBy(col("timestamp").desc())

df_dtcs_clean = (
    df_raw_dtcs
    .withColumn("_rn", row_number().over(w_dtc))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .withColumn("dtc_code", upper(trim(col("dtc_code"))))
    .withColumn("severity_rank",
        when(col("severity") == "Critical", lit(4))
        .when(col("severity") == "High", lit(3))
        .when(col("severity") == "Medium", lit(2))
        .when(col("severity") == "Low", lit(1))
        .otherwise(lit(0)))
    .withColumn("_cleaned_at", current_timestamp())
)

df_dtcs_clean.write.format("delta").mode("overwrite").save("Tables/fact_dtc_events")
print(f"Silver fact_dtc_events: {df_dtcs_clean.count()} rows")

## Clean Maintenance Records

In [ ]:
# Deduplicate by maintenance_id
w_maint = Window.partitionBy("maintenance_id").orderBy(col("service_date").desc())

df_maint_clean = (
    df_raw_maintenance
    .withColumn("_rn", row_number().over(w_maint))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .withColumn("cost_usd", when(col("cost_usd") < 0, lit(0)).otherwise(col("cost_usd")))
    .withColumn("labor_hours", when(col("labor_hours") < 0, lit(0)).otherwise(col("labor_hours")))
    .withColumn("service_date_only", to_date(col("service_date")))
    .withColumn("_cleaned_at", current_timestamp())
)

df_maint_clean.write.format("delta").mode("overwrite").save("Tables/fact_maintenance")
print(f"Silver fact_maintenance: {df_maint_clean.count()} rows")

## Summary

In [ ]:
print("=== Silver Layer Transformation Complete ===")
print(f"  dim_vehicles:     {spark.read.format('delta').load('Tables/dim_vehicles').count()} rows")
print(f"  fact_telemetry:   {spark.read.format('delta').load('Tables/fact_telemetry').count()} rows")
print(f"  fact_dtc_events:  {spark.read.format('delta').load('Tables/fact_dtc_events').count()} rows")
print(f"  fact_maintenance: {spark.read.format('delta').load('Tables/fact_maintenance').count()} rows")